In [1]:
import numpy as np 
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt 
import warnings 
warnings.filterwarnings('ignore')


In [2]:
df = pd.read_csv("movies_metadata.csv")

In [3]:
df.head()
# df.columns
# df.info

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [4]:
df.shape 

(45466, 24)

In [5]:
df.isnull().sum()

adult                        0
belongs_to_collection    40972
budget                       0
genres                       0
homepage                 37684
id                           0
imdb_id                     17
original_language           11
original_title               0
overview                   954
popularity                   5
poster_path                386
production_companies         3
production_countries         3
release_date                87
revenue                      6
runtime                    263
spoken_languages             6
status                      87
tagline                  25054
title                        6
video                        6
vote_average                 6
vote_count                   6
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(13)

In [7]:
df = df.drop_duplicates().reset_index(drop = True)

In [8]:
df = df[['title','overview', 'genres','tagline','vote_average','popularity']]

In [9]:
df.isnull().sum()

title               6
overview          954
genres              0
tagline         25045
vote_average        6
popularity          5
dtype: int64

In [10]:
df = df.dropna(subset=['title'])

In [11]:
df ['overview']= df['overview'].fillna(" ")

In [12]:
df.iloc[0]['genres']
import ast 

In [13]:
df['genres'] = df['genres'].apply(lambda x:" ".join([i['name'] for i in ast.literal_eval(x)]))

In [14]:
df['tagline'] = df['tagline'].fillna(' ')

In [15]:
df.isnull().sum()

title           0
overview        0
genres          0
tagline         0
vote_average    0
popularity      0
dtype: int64

In [16]:
df['tags'] = df['overview']+ " "+df['genres']+" "+ df['tagline']

In [17]:
df['tags'][0]

"Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences. Animation Comedy Family  "

In [18]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re 

In [19]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [20]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [21]:
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]',"",text)

    words = text.split()

    words = [word for word in words if word not in stop_words]

    words =[lemmatizer.lemmatize(word) for word in words]


    return " ".join(words)



In [22]:
df['tags']=df['tags'].apply(preprocess_text)

In [23]:
df['tags'][1]

'sibling judy peter discover enchanted board game open door magical world unwittingly invite alan adult who trapped inside game year living room alans hope freedom finish game prof risky three find running giant rhinoceros evil monkey terrifying creature adventure fantasy family roll dice unleash excitement'

In [24]:
df = df.reset_index(drop=True)

In [25]:
indices = pd.Series(df.index , index=df['title']).drop_duplicates()
indices


title
Toy Story                          0
Jumanji                            1
Grumpier Old Men                   2
Waiting to Exhale                  3
Father of the Bride Part II        4
                               ...  
Subdue                         45442
Century of Birthing            45443
Betrayal                       45444
Satan Triumphant               45445
Queerama                       45446
Length: 45447, dtype: int64

In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer


In [27]:
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,2), stop_words='english')

In [28]:
tfidf_matrix =tfidf.fit_transform(df['tags'])

In [29]:
tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1548114 stored elements and shape (45447, 50000)>

In [30]:
from sklearn.metrics.pairwise import cosine_similarity



In [31]:
def recommend(title, n=10):
    global indices, tfidf_matrix, tfidf, df
    for _c in ['title','overview','genres','tagline']:
        if _c not in df.columns:
            df[_c] = ''
    if 'tags' not in df.columns:
        import ast, re
        def genres_to_names(x):
            try:
                if pd.isna(x) or str(x).strip()=='':
                    return ''
                return ' '.join([i.get('name','') for i in ast.literal_eval(x) if isinstance(i, dict)])
            except Exception:
                return str(x)
        df['genres'] = df['genres'].apply(genres_to_names)
        df['tags'] = (df['overview'].fillna('') + ' ' + df['genres'].fillna('') + ' ' + df['tagline'].fillna(''))
        df['tags'] = df['tags'].astype(str).str.lower().map(lambda t: re.sub(r'[^a-z\s]',' ',t))
        df['tags'] = df['tags'].str.replace(r'\s+',' ', regex=True).str.strip()

    try:
        indices
    except NameError:
        df.reset_index(drop=True, inplace=True)
        indices = pd.Series(df.index, index=df['title']).drop_duplicates()
    
    try:
        tfidf_matrix
    except NameError:
        from sklearn.feature_extraction.text import TfidfVectorizer
        tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,2), stop_words='english')
        tfidf_matrix = tfidf.fit_transform(df['tags'].astype('str'))

    if title not in indices.index:
        matches = [t for t in indices.index if str(t).lower() == str(title).lower()]
        if matches:
            title = matches[0]
        else:
            return ['Movie not found']
    idx = int(indices[title])
    from sklearn.metrics.pairwise import cosine_similarity
    sim_score = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    similar_idx = sim_score.argsort()[::-1][1:n+1]
    return list(df['title'].iloc[similar_idx])



In [32]:
recommend('Avenger')

["Dexter's Laboratory: Ego Trip",
 'D.O.A.',
 'Hit by Lightning',
 'The Hunting Party',
 'The Philadelphia Story',
 'Hopscotch',
 "Now You See Him, Now You Don't",
 'Tough and Deadly',
 'The Escapist',
 'Sharpshooter']

In [33]:
import pickle 
pickle.dump(tfidf_matrix, open('tfidf_matrix.pkl','wb'))
pickle.dump(indices, open('indices.pkl','wb'))

df.to_pickle('df.pkl')
pickle.dump(tfidf, open('tfidf.pkl','wb'))